# DyslexiaLens — Multi-Modal, Multi-Task CNN Pipeline (v2)
## EMNIST-GAMBO · Late Fusion · Dual-Head · Stability Fixes
---
**Architecture:** Keras Functional API · Custom ACN Layer · Late Fusion
**Inputs:** Image tensor (128×128×1) + 6 engineered clinical features
**Outputs:** Classification head (sigmoid) + Severity score head (sigmoid + masked Huber)
**Dataset:** EMNIST-GAMBO Balanced (~204k samples) — 1:1 class ratio

**v2 Fixes applied from run-1 analysis:**
1. `LEARNING_RATE` reduced `1e-3 → 3e-4` — eliminates val accuracy oscillation (±33pp swing)
2. Masked Huber loss on severity head — control samples (class=0) are zero-masked so the head learns the correct floor of 0.0
3. Dual `ModelCheckpoint` — one for best val accuracy, one for best val severity MAE, so the saved checkpoint is never the worst MAE state
4. `EarlyStopping` now monitors `val_loss` (composite) instead of accuracy alone — prevents the epoch-19 spike scenario


## 1. Imports & Environment Setup

In [ ]:
import os, re, zipfile, pathlib, warnings, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from PIL import Image
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.losses import Huber
import tensorflow.keras.backend as K

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow  : {tf.__version__}")
print(f"Keras       : {keras.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"GPU devices : {tf.config.list_physical_devices('GPU')}")


## 2. Global Configuration

In [ ]:
# ── Kaggle Paths ──────────────────────────────────────────────────────────────
KAGGLE_INPUT = "/kaggle/input/datasets/adhyatmaaaa/gambo-emnist"
IMG_ROOT     = f"{KAGGLE_INPUT}/Dataset_Gambo_EMNIST (1)"
CSV_EMNIST   = f"{KAGGLE_INPUT}/Dataset_Dyslexia_EMNIST_FeatureEngineeringV2.csv"
CSV_NOAUG    = f"{KAGGLE_INPUT}/Dataset_Dyslexia_NoAugmentation_FeatureEngineeringV2.csv"

WORKING_DIR  = "/kaggle/working"
os.makedirs(f"{WORKING_DIR}/checkpoints", exist_ok=True)

# Dual checkpoints — one per head objective (FIX 3)
CKPT_ACC_PATH  = f"{WORKING_DIR}/checkpoints/best_clf_acc.weights.h5"
CKPT_MAE_PATH  = f"{WORKING_DIR}/checkpoints/best_sev_mae.weights.h5"
MODEL_SAVE_PATH  = f"{WORKING_DIR}/dyslexialens_v2_model.keras"
SCALER_SAVE_PATH = f"{WORKING_DIR}/feature_scaler.pkl"
LOG_DIR = f"{WORKING_DIR}/logs/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
os.makedirs(LOG_DIR, exist_ok=True)

# ── Image config ──────────────────────────────────────────────────────────────
IMG_SIZE        = (128, 128)
IMG_CHANNELS    = 1
INPUT_SHAPE_IMG = (*IMG_SIZE, IMG_CHANNELS)

# ── Feature config ────────────────────────────────────────────────────────────
FEATURE_COLS = [
    "stroke_density",
    "center_of_mass_x",
    "center_of_mass_y",
    "bounding_box_ratio",
    "stroke_transitions",
    "horizontal_symmetry",
]
N_FEATURES = len(FEATURE_COLS)

# ── Severity proxy weights ────────────────────────────────────────────────────
SEV_W_SYMMETRY    = 0.55
SEV_W_TRANSITIONS = 0.30
SEV_W_BBOX        = 0.15

# ── Training hyperparameters ──────────────────────────────────────────────────
BATCH_SIZE    = 64
EPOCHS        = 50
SEED          = 42
DROPOUT_RATE  = 0.40

# FIX 1 — Learning rate reduced 1e-3 → 3e-4
# Run-1 showed val accuracy oscillating ±33pp (std=0.082) with LR=1e-3.
# 3e-4 is the standard stable starting point for Adam on large datasets.
LEARNING_RATE = 3e-4

# ── Loss weights (multi-task) ─────────────────────────────────────────────────
LOSS_WEIGHT_CLS = 1.0
LOSS_WEIGHT_SEV = 0.5

# ── Threshold sweep ───────────────────────────────────────────────────────────
THRESHOLD_SWEEP_RANGE = np.arange(0.30, 0.71, 0.01)
DEFAULT_THRESHOLD     = 0.40

# ── BinaryFocalLoss guard (BLACKLISTED) ───────────────────────────────────────
USE_FOCAL_LOSS = False
assert not USE_FOCAL_LOSS, "BinaryFocalLoss is blacklisted for DyslexiaLens."

assert os.path.isdir(IMG_ROOT),    f"IMG_ROOT not found: {IMG_ROOT}"
assert os.path.isfile(CSV_EMNIST), f"CSV not found: {CSV_EMNIST}"

print("Configuration loaded.")
print(f"  KAGGLE_INPUT      : {KAGGLE_INPUT}")
print(f"  IMG_ROOT          : {IMG_ROOT}")
print(f"  Image input shape : {INPUT_SHAPE_IMG}")
print(f"  Feature vector dim: {N_FEATURES}  →  {FEATURE_COLS}")
print(f"  Learning rate     : {LEARNING_RATE}  ← reduced from 1e-3 (v1 instability fix)")
print(f"  Loss weights      : clf={LOSS_WEIGHT_CLS}, sev={LOSS_WEIGHT_SEV}")
print(f"  Operational thresh: {DEFAULT_THRESHOLD}")


## 3. Dataset Inventory

In [ ]:
print(f"Verifying dataset structure under: {IMG_ROOT}\n")

total = 0
for split in ["Train", "Validation", "Test"]:
    for cat in ["Corrected", "Normal", "Reversal"]:
        folder = pathlib.Path(IMG_ROOT) / split / cat
        if folder.exists():
            count = len(list(folder.glob("*.png")))
            total += count
            print(f"  {split:12s} / {cat:10s} : {count:,} images")
        else:
            print(f"  [MISSING] {split}/{cat}")

print(f"\nTotal .png files verified: {total:,}")
print(f"Expected ~204,833 for EMNIST-balanced dataset.")


## 4. Feature CSV Loading & Severity Proxy Construction

In [ ]:
df_raw = pd.read_csv(CSV_EMNIST)
print(f"Raw CSV shape    : {df_raw.shape}")
print(f"Class balance    : {df_raw['target_class'].value_counts().to_dict()}")
print(f"Splits           : {df_raw['split'].value_counts().to_dict()}")
print(f"NaN check        : {df_raw.isnull().sum().sum()} total NaNs")


In [ ]:
# ── 4.2  Vectorized path construction (9x faster than row-wise apply) ─────────
# Uses file_name, split, folder_category columns directly — zero parsing needed.
df_raw['local_path'] = (
    IMG_ROOT + '/' +
    df_raw['split'] + '/' +
    df_raw['folder_category'] + '/' +
    df_raw['file_name']
)

print("Path resolution spot-check:")
for p in df_raw['local_path'].head(5):
    exists = os.path.exists(p)
    print(f"  {'✓' if exists else '✗'}  {p}")

sample_idx = [0, 1000, 50000, 143000, 204832]
n_missing = sum(1 for i in sample_idx if not os.path.exists(df_raw['local_path'].iloc[i]))
print(f"\nSpot-check ({len(sample_idx)} samples): {'✓ All exist' if n_missing == 0 else f'⚠ {n_missing} missing'}")
print(f"Total paths built: {len(df_raw):,}")


In [ ]:
# ── 4.3  Severity Proxy Construction ─────────────────────────────────────────
# Control group  (target_class == 0) → severity = 0.0  (no clinical indicators)
# Dyslexia group (target_class == 1) → normalized weighted deviation index ∈ [0,1]

def build_severity_proxy(df: pd.DataFrame) -> pd.Series:
    sev = pd.Series(0.0, index=df.index, dtype=np.float32)
    mask = df['target_class'] == 1
    sub  = df.loc[mask].copy()

    def minmax(s: pd.Series) -> pd.Series:
        lo, hi = s.min(), s.max()
        return (s - lo) / (hi - lo + 1e-8)

    asym  = minmax(1.0 - sub['horizontal_symmetry'])
    trans = minmax(sub['stroke_transitions'])
    bbox  = minmax(sub['bounding_box_ratio'])

    raw = (SEV_W_SYMMETRY * asym + SEV_W_TRANSITIONS * trans + SEV_W_BBOX * bbox)
    sev.loc[mask] = minmax(raw).values.astype(np.float32)
    return sev

df_raw['severity_proxy'] = build_severity_proxy(df_raw)

print("Severity proxy statistics:")
print(f"  Control  (0) mean  : {df_raw.loc[df_raw['target_class']==0,'severity_proxy'].mean():.4f}  ← must be 0.0")
print(f"  Dyslexia (1) mean  : {df_raw.loc[df_raw['target_class']==1,'severity_proxy'].mean():.4f}")
print(f"  Dyslexia (1) std   : {df_raw.loc[df_raw['target_class']==1,'severity_proxy'].std():.4f}")
print(f"  Dyslexia (1) range : [{df_raw.loc[df_raw['target_class']==1,'severity_proxy'].min():.4f}, "
      f"{df_raw.loc[df_raw['target_class']==1,'severity_proxy'].max():.4f}]")


## 5. Feature Scaling & Dataset Splits

In [ ]:
df_train = df_raw[df_raw['split'] == 'Train'].reset_index(drop=True)
df_val   = df_raw[df_raw['split'] == 'Validation'].reset_index(drop=True)
df_test  = df_raw[df_raw['split'] == 'Test'].reset_index(drop=True)

print(f"Train      : {len(df_train):,}  (class 0: {(df_train['target_class']==0).sum():,} | class 1: {(df_train['target_class']==1).sum():,})")
print(f"Validation : {len(df_val):,}  (class 0: {(df_val['target_class']==0).sum():,} | class 1: {(df_val['target_class']==1).sum():,})")
print(f"Test       : {len(df_test):,}  (class 0: {(df_test['target_class']==0).sum():,} | class 1: {(df_test['target_class']==1).sum():,})")


In [ ]:
scaler = MinMaxScaler()
scaler.fit(df_train[FEATURE_COLS])

X_feat_train = scaler.transform(df_train[FEATURE_COLS]).astype(np.float32)
X_feat_val   = scaler.transform(df_val[FEATURE_COLS]).astype(np.float32)
X_feat_test  = scaler.transform(df_test[FEATURE_COLS]).astype(np.float32)

y_cls_train = df_train['target_class'].values.astype(np.float32)
y_cls_val   = df_val['target_class'].values.astype(np.float32)
y_cls_test  = df_test['target_class'].values.astype(np.float32)

y_sev_train = df_train['severity_proxy'].values.astype(np.float32)
y_sev_val   = df_val['severity_proxy'].values.astype(np.float32)
y_sev_test  = df_test['severity_proxy'].values.astype(np.float32)

print(f"Feature matrix — Train: {X_feat_train.shape} | Val: {X_feat_val.shape} | Test: {X_feat_test.shape}")
print(f"Scaler range: {scaler.data_min_.round(3)} → {scaler.data_max_.round(3)}")


In [ ]:
class_weights_array = compute_class_weight(
    class_weight='balanced', classes=np.array([0, 1]), y=y_cls_train
)
CLASS_WEIGHTS = {0: float(class_weights_array[0]), 1: float(class_weights_array[1])}
print(f"Class weights: {CLASS_WEIGHTS}")

sample_weights_train = np.where(
    y_cls_train == 1, CLASS_WEIGHTS[1], CLASS_WEIGHTS[0]
).astype(np.float32)

print(f"Sample weight shape : {sample_weights_train.shape}")
print(f"Mean weight (train) : {sample_weights_train.mean():.4f}  (should be ~1.0)")


## 6. tf.data Input Pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess_image(path: str) -> tf.Tensor:
    raw = tf.io.read_file(path)
    img = tf.image.decode_png(raw, channels=IMG_CHANNELS)
    img = tf.image.resize(img, IMG_SIZE, method='bilinear')
    img = tf.cast(img, tf.float32) / 255.0
    return img

def augment_image(img: tf.Tensor) -> tf.Tensor:
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.08)
    img = tf.image.random_contrast(img, lower=0.92, upper=1.08)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img

def make_dataset(
    paths, features, y_cls, y_sev,
    sample_weights=None, batch_size=BATCH_SIZE,
    training=False, shuffle_buffer=2048
) -> tf.data.Dataset:
    path_ds = tf.data.Dataset.from_tensor_slices(paths)
    feat_ds = tf.data.Dataset.from_tensor_slices(features)
    cls_ds  = tf.data.Dataset.from_tensor_slices(y_cls)
    sev_ds  = tf.data.Dataset.from_tensor_slices(y_sev)

    img_ds = path_ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    if training:
        img_ds = img_ds.map(augment_image, num_parallel_calls=AUTOTUNE)

    sw_ds = tf.data.Dataset.from_tensor_slices(
        sample_weights.astype(np.float32) if sample_weights is not None
        else np.ones(len(y_cls), dtype=np.float32)
    )

    ds = tf.data.Dataset.zip((
        tf.data.Dataset.zip((img_ds, feat_ds)),
        tf.data.Dataset.zip((cls_ds, sev_ds)),
        sw_ds
    ))
    ds = ds.map(
        lambda inputs, targets, sw: (
            {'image_input': inputs[0], 'feature_input': inputs[1]},
            {'classification_head': targets[0], 'severity_head': targets[1]},
            sw
        ),
        num_parallel_calls=AUTOTUNE
    )
    if training:
        ds = ds.shuffle(shuffle_buffer, seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(AUTOTUNE)


train_ds = make_dataset(df_train['local_path'].values, X_feat_train, y_cls_train, y_sev_train,
                        sample_weights=sample_weights_train, training=True)
val_ds   = make_dataset(df_val['local_path'].values,   X_feat_val,   y_cls_val,   y_sev_val,   training=False)
test_ds  = make_dataset(df_test['local_path'].values,  X_feat_test,  y_cls_test,  y_sev_test,  training=False)

for inputs, targets, sw in train_ds.take(1):
    print("Image input shape   :", inputs['image_input'].shape)
    print("Feature input shape :", inputs['feature_input'].shape)
    print("Classification y    :", targets['classification_head'].shape)
    print("Severity y          :", targets['severity_head'].shape)
    print("Sample weights      :", sw.shape, " sample:", sw[:4].numpy())


## 7. Custom AdaptiveContrastNorm (ACN) Layer

In [ ]:
class AdaptiveContrastNorm(layers.Layer):
    """
    AdaptiveContrastNorm (ACN) — proven custom layer from DyslexiaLens ablation study.

    Per-sample, per-spatial-region contrast normalisation applied before any
    learnable convolution weights. Instance-independent (unlike BatchNorm),
    making it stable at variable inference batch sizes.

    Mechanism:
        μ  = mean(x)  over H,W per sample
        σ  = std(x) + ε
        x̂  = (x − μ) / σ
        y  = γ * x̂ + β   (learnable affine, initialised to identity)
    """
    def __init__(self, epsilon: float = 1e-6, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon

    def build(self, input_shape):
        channels = input_shape[-1]
        self.gamma = self.add_weight('gamma', shape=(1,1,1,channels), initializer='ones', trainable=True)
        self.beta  = self.add_weight('beta',  shape=(1,1,1,channels), initializer='zeros', trainable=True)
        super().build(input_shape)

    def call(self, x, training=None):
        mu    = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        sigma = tf.math.reduce_std(x, axis=[1, 2], keepdims=True) + self.epsilon
        return self.gamma * ((x - mu) / sigma) + self.beta

    def get_config(self):
        config = super().get_config()
        config.update({'epsilon': self.epsilon})
        return config


## 8. Model Architecture — Late Fusion + Dual-Head (Functional API)

In [ ]:
def build_dyslexialens_model(
    img_shape=INPUT_SHAPE_IMG,
    n_features=N_FEATURES,
    dropout_rate=DROPOUT_RATE
) -> Model:
    """
    DyslexiaLens v2 — Multi-Modal, Multi-Task Model (Keras Functional API).

    Image Branch: ACN → Conv2D(32)→BN→Pool → Conv2D(64)→BN→Pool →
                  Conv2D(128)→BN→Pool → Conv2D(256)→BN→GlobalAvgPool
    Feature Branch: Dense(32)→BN → Dense(64)
    Fusion: Concatenate → Dense(256)→BN→Dropout → Dense(128)→Dropout
    Head 1 (classification_head): Dense(1, sigmoid)
    Head 2 (severity_head):       Dense(1, sigmoid)
    """
    image_input = Input(shape=img_shape, name='image_input')
    x = AdaptiveContrastNorm(name='acn')(image_input)

    x = layers.Conv2D(32,  (3,3), padding='same', activation='relu', name='conv1')(x)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.MaxPooling2D((2,2), name='pool1')(x)
    x = layers.Dropout(dropout_rate * 0.5, name='drop1')(x)

    x = layers.Conv2D(64,  (3,3), padding='same', activation='relu', name='conv2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.MaxPooling2D((2,2), name='pool2')(x)
    x = layers.Dropout(dropout_rate * 0.5, name='drop2')(x)

    x = layers.Conv2D(128, (3,3), padding='same', activation='relu', name='conv3')(x)
    x = layers.BatchNormalization(name='bn3')(x)
    x = layers.MaxPooling2D((2,2), name='pool3')(x)
    x = layers.Dropout(dropout_rate * 0.5, name='drop3')(x)

    x = layers.Conv2D(256, (3,3), padding='same', activation='relu', name='conv4')(x)
    x = layers.BatchNormalization(name='bn4')(x)
    image_embedding = layers.GlobalAveragePooling2D(name='image_embedding')(x)

    feature_input = Input(shape=(n_features,), name='feature_input')
    f = layers.Dense(32,  activation='relu', name='feat_dense1')(feature_input)
    f = layers.BatchNormalization(name='feat_bn1')(f)
    feature_embedding = layers.Dense(64, activation='relu', name='feature_embedding')(f)

    fused = layers.Concatenate(name='late_fusion')([image_embedding, feature_embedding])
    fused = layers.Dense(256, activation='relu', name='fusion_dense1')(fused)
    fused = layers.BatchNormalization(name='fusion_bn1')(fused)
    fused = layers.Dropout(dropout_rate, name='fusion_drop1')(fused)
    fused = layers.Dense(128, activation='relu', name='fusion_dense2')(fused)
    fused = layers.Dropout(dropout_rate * 0.5, name='fusion_drop2')(fused)

    classification_head = layers.Dense(1, activation='sigmoid', name='classification_head')(fused)
    severity_head       = layers.Dense(1, activation='sigmoid', name='severity_head')(fused)

    return Model(inputs=[image_input, feature_input],
                 outputs=[classification_head, severity_head],
                 name='DyslexiaLens_LateFusion_v2')

model = build_dyslexialens_model()
model.summary(line_length=100, expand_nested=True)


## 9. Custom Masked Severity Loss + Model Compilation

In [ ]:
# ── FIX 2: Masked Huber loss for the severity head ───────────────────────────
#
# Root cause of MAE=0.424 in run-1:
#   The severity head was outputting ~0.14 for control samples (target=0.0)
#   because standard Huber loss penalises ALL samples equally, including
#   control samples where the target is 0.0. Since control samples have no
#   clinical markers driving the feature branch toward 0.0, the head settles
#   at a non-zero floor.
#
# Fix:
#   Compute Huber loss ONLY on dyslexia samples (target_class == 1).
#   Control samples contribute zero gradient to the severity head.
#   This forces the head to learn the correct 0.0 floor.
#
# Implementation:
#   mask = (y_true > 0).float()   — 1 for dyslexia, 0 for control
#   loss = Huber(y_true, y_pred) * mask
#   mean over dyslexia samples only (sum / (sum(mask) + ε))

class MaskedHuberLoss(keras.losses.Loss):
    """
    Huber loss applied only to dyslexia samples (severity_proxy > 0).
    Control samples (severity_proxy == 0.0) are zero-masked — they
    contribute no gradient to the severity head.
    """
    def __init__(self, delta: float = 0.5, **kwargs):
        super().__init__(**kwargs)
        self.delta   = delta
        self.huber   = Huber(delta=delta, reduction='none')

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1, 1]), tf.float32)
        y_pred = tf.cast(tf.reshape(y_pred, [-1, 1]), tf.float32)

        # mask: 1.0 where sample is dyslexia (target > 0), 0.0 for control
        mask = tf.cast(y_true > 0.0, tf.float32)

        per_sample_loss = self.huber(y_true, y_pred)
        masked_loss     = per_sample_loss * tf.squeeze(mask, axis=-1)

        # Normalise by number of dyslexia samples in the batch (not total batch size)
        n_dyslexia = tf.reduce_sum(mask) + 1e-8
        return tf.reduce_sum(masked_loss) / n_dyslexia

    def get_config(self):
        config = super().get_config()
        config.update({'delta': self.delta})
        return config


model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss={
        'classification_head': keras.losses.BinaryCrossentropy(),
        'severity_head':       MaskedHuberLoss(delta=0.5),    # FIX 2
    },
    loss_weights={
        'classification_head': LOSS_WEIGHT_CLS,
        'severity_head':       LOSS_WEIGHT_SEV,
    },
    metrics={
        'classification_head': [
            keras.metrics.BinaryAccuracy(name='accuracy'),
            keras.metrics.AUC(name='auc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
        ],
        'severity_head': [
            keras.metrics.MeanAbsoluteError(name='mae'),
            keras.metrics.RootMeanSquaredError(name='rmse'),
        ],
    }
)
print("Model compiled.")
print(f"  Optimiser       : Adam (lr={LEARNING_RATE})  ← reduced from 1e-3")
print(f"  Clf loss        : BinaryCrossentropy  (weight={LOSS_WEIGHT_CLS})")
print(f"  Severity loss   : MaskedHuberLoss(delta=0.5)  (weight={LOSS_WEIGHT_SEV})")
print(f"  [GUARD] BinaryFocalLoss : DISABLED")


## 10. Training Callbacks

In [ ]:
# ── FIX 3: Dual ModelCheckpoint ───────────────────────────────────────────────
# Run-1 problem: a single checkpoint monitored val_acc, so it saved epoch 19
# (best acc=97.0%) which happened to have the worst severity MAE=0.424 in the run.
#
# Fix: two independent checkpoints — one per head objective.
# After training, we compare both and load the one with the best joint result.

# ── FIX 4: EarlyStopping monitors val_loss (composite) ───────────────────────
# Run-1 problem: EarlyStopping monitored val_classification_head_accuracy.
# When the severity head spiked at epoch 19, val_acc was at its best, so
# early stopping was blind to the severity degradation.
#
# val_loss is the weighted sum of both head losses — it reflects degradation
# in EITHER head, preventing us from locking in a spike state.

callbacks = [
    # Monitor composite val_loss — catches spikes in either head
    EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        mode='min',
        verbose=1
    ),
    # Halve LR when val_loss plateaus for 4 epochs
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        mode='min',
        verbose=1
    ),
    # Checkpoint 1: best classification accuracy
    ModelCheckpoint(
        filepath=CKPT_ACC_PATH,
        monitor='val_classification_head_accuracy',
        save_best_only=True,
        save_weights_only=True,
        mode='max',
        verbose=1
    ),
    # Checkpoint 2: best severity MAE
    ModelCheckpoint(
        filepath=CKPT_MAE_PATH,
        monitor='val_severity_head_mae',
        save_best_only=True,
        save_weights_only=True,
        mode='min',
        verbose=1
    ),
]
print("Callbacks registered:")
for cb in callbacks:
    monitor = getattr(cb, 'monitor', '—')
    print(f"  • {cb.__class__.__name__:<22s} monitors: {monitor}")


## 11. Model Training

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print(f"\nTraining complete.")
print(f"  Total epochs run : {len(history.history['loss'])}")
print(f"  Best val clf acc : {max(history.history['val_classification_head_accuracy']):.4f}")
print(f"  Best val sev MAE : {min(history.history['val_severity_head_mae']):.4f}")
print(f"  Best val loss    : {min(history.history['val_loss']):.4f}")


## 12. Training History Visualisation

In [ ]:
def plot_training_history(history):
    h = history.history
    epochs_ran = range(1, len(h['loss']) + 1)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('DyslexiaLens v2 — Training History', fontsize=15, fontweight='bold')

    ax = axes[0, 0]
    ax.plot(epochs_ran, h['loss'],     label='Train Loss', color='steelblue')
    ax.plot(epochs_ran, h['val_loss'], label='Val Loss',   color='coral', linestyle='--')
    ax.set_title('Total Combined Loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[0, 1]
    ax.plot(epochs_ran, h['classification_head_accuracy'],     label='Train Acc', color='steelblue')
    ax.plot(epochs_ran, h['val_classification_head_accuracy'], label='Val Acc',   color='coral', linestyle='--')
    ax.set_title('Classification Head — Accuracy')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[0, 2]
    ax.plot(epochs_ran, h['classification_head_auc'],     label='Train AUC', color='steelblue')
    ax.plot(epochs_ran, h['val_classification_head_auc'], label='Val AUC',   color='coral', linestyle='--')
    ax.set_title('Classification Head — AUC-ROC')
    ax.set_xlabel('Epoch'); ax.set_ylabel('AUC')
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1, 0]
    ax.plot(epochs_ran, h['severity_head_mae'],     label='Train MAE', color='mediumseagreen')
    ax.plot(epochs_ran, h['val_severity_head_mae'], label='Val MAE',   color='orangered', linestyle='--')
    ax.axhline(0.0214, color='gray', linestyle=':', alpha=0.7, label='Baseline MAE=0.0214')
    ax.set_title('Severity Head — MAE (Masked Huber)')
    ax.set_xlabel('Epoch'); ax.set_ylabel('MAE')
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1, 1]
    ax.plot(epochs_ran, h['severity_head_rmse'],     label='Train RMSE', color='mediumseagreen')
    ax.plot(epochs_ran, h['val_severity_head_rmse'], label='Val RMSE',   color='orangered', linestyle='--')
    ax.set_title('Severity Head — RMSE')
    ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE')
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1, 2]
    ax.plot(epochs_ran, h['classification_head_precision'],     label='Train Prec', color='steelblue')
    ax.plot(epochs_ran, h['val_classification_head_precision'], label='Val Prec',   color='coral', linestyle='--')
    ax.plot(epochs_ran, h['classification_head_recall'],        label='Train Rec',  color='mediumseagreen')
    ax.plot(epochs_ran, h['val_classification_head_recall'],    label='Val Rec',    color='orangered', linestyle='--')
    ax.set_title('Classification Head — Precision & Recall')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Score')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('/kaggle/working/training_history_v2.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved → /kaggle/working/training_history_v2.png")

plot_training_history(history)


## 13. Checkpoint Selection & Threshold Sweep

In [ ]:
# ── Select the best checkpoint across both saved objectives ───────────────────
#
# We have two saved weight files:
#   CKPT_ACC_PATH — weights at epoch with best val classification accuracy
#   CKPT_MAE_PATH — weights at epoch with best val severity MAE
#
# Strategy: load each, evaluate on val_ds for both metrics, pick the one
# with the better joint score: acc + (1 - normalized_MAE).

def evaluate_checkpoint(ckpt_path, label):
    model.load_weights(ckpt_path)
    probs_cls, probs_sev, true_cls, true_sev = [], [], [], []
    for inputs, targets, _ in val_ds:
        p_cls, p_sev = model(inputs, training=False)
        probs_cls.append(p_cls.numpy().flatten())
        probs_sev.append(p_sev.numpy().flatten())
        true_cls.append(targets['classification_head'].numpy().flatten())
        true_sev.append(targets['severity_head'].numpy().flatten())
    probs_cls = np.concatenate(probs_cls)
    probs_sev = np.concatenate(probs_sev)
    true_cls  = np.concatenate(true_cls).astype(int)
    true_sev  = np.concatenate(true_sev)
    acc = np.mean((probs_cls >= 0.5).astype(int) == true_cls)
    mae = np.mean(np.abs(probs_sev - true_sev))
    print(f"  {label:<30s} → val_acc={acc:.4f} | val_sev_MAE={mae:.4f}")
    return acc, mae, probs_cls, probs_sev, true_cls, true_sev

print("Evaluating saved checkpoints on validation set:")
acc_a, mae_a, *_ = evaluate_checkpoint(CKPT_ACC_PATH, "best_clf_acc checkpoint")
acc_m, mae_m, probs_cls, probs_sev, true_cls, true_sev = evaluate_checkpoint(CKPT_MAE_PATH, "best_sev_mae checkpoint")

# Joint score: penalise MAE relative to a target of 0.05 (reasonable ceiling)
MAE_TARGET = 0.05
score_a = acc_a - max(0, mae_a - MAE_TARGET)
score_m = acc_m - max(0, mae_m - MAE_TARGET)

if score_a >= score_m:
    print(f"\n► Using best_clf_acc checkpoint (joint score {score_a:.4f} ≥ {score_m:.4f})")
    model.load_weights(CKPT_ACC_PATH)
    # re-collect val predictions with this checkpoint
    _, _, probs_cls, probs_sev, true_cls, true_sev = evaluate_checkpoint(CKPT_ACC_PATH, "final checkpoint")
else:
    print(f"\n► Using best_sev_mae checkpoint (joint score {score_m:.4f} > {score_a:.4f})")
    # already loaded above


In [ ]:
# ── Threshold sweep on val set ───────────────────────────────────────────────
val_probs_cls = probs_cls
val_probs_sev = probs_sev
val_true_cls  = true_cls
val_true_sev  = true_sev

sweep_results = []
for t in THRESHOLD_SWEEP_RANGE:
    preds = (val_probs_cls >= t).astype(int)
    acc   = np.mean(preds == val_true_cls)
    tp    = np.sum((preds == 1) & (val_true_cls == 1))
    fp    = np.sum((preds == 1) & (val_true_cls == 0))
    fn    = np.sum((preds == 0) & (val_true_cls == 1))
    prec  = tp / (tp + fp + 1e-8)
    rec   = tp / (tp + fn + 1e-8)
    f1    = 2 * prec * rec / (prec + rec + 1e-8)
    sweep_results.append({'threshold': t, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1})

sweep_df = pd.DataFrame(sweep_results)
best_row = sweep_df.loc[sweep_df['accuracy'].idxmax()]
OPTIMAL_THRESHOLD = best_row['threshold']

print("Threshold Sweep Results:")
print(sweep_df.to_string(index=False, float_format='{:.4f}'.format))
print(f"\n  ► Optimal threshold : t = {OPTIMAL_THRESHOLD:.2f}")
print(f"    Accuracy @ t*     : {best_row['accuracy']:.4f}")
print(f"    F1 @ t*           : {best_row['f1']:.4f}")
print(f"    [BASELINE] t=0.40 → Acc=97.86%, MAE=0.0214")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Threshold Sweep — Classification Head (v2)', fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(sweep_df['threshold'], sweep_df['accuracy'],  label='Accuracy',  color='steelblue')
ax.plot(sweep_df['threshold'], sweep_df['f1'],        label='F1 Score',  color='mediumseagreen')
ax.plot(sweep_df['threshold'], sweep_df['precision'], label='Precision', color='coral')
ax.plot(sweep_df['threshold'], sweep_df['recall'],    label='Recall',    color='orchid')
ax.axvline(OPTIMAL_THRESHOLD, color='black', linestyle='--', label=f't* = {OPTIMAL_THRESHOLD:.2f}')
ax.axvline(DEFAULT_THRESHOLD, color='gray',  linestyle=':',  label=f'Baseline t = {DEFAULT_THRESHOLD:.2f}', alpha=0.7)
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Metrics vs. Decision Threshold')
ax.legend(); ax.grid(alpha=0.3)

fpr, tpr, _ = roc_curve(val_true_cls, val_probs_cls)
auc_score   = roc_auc_score(val_true_cls, val_probs_cls)
ax = axes[1]
ax.plot(fpr, tpr, color='steelblue', label=f'ROC (AUC = {auc_score:.4f})')
ax.plot([0,1],[0,1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Validation Set')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/threshold_sweep_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → /kaggle/working/threshold_sweep_v2.png")


## 14. Final Evaluation on Test Set

In [ ]:
test_probs_cls, test_probs_sev, test_true_cls, test_true_sev = [], [], [], []

for inputs, targets, _ in test_ds:
    p_cls, p_sev = model(inputs, training=False)
    test_probs_cls.append(p_cls.numpy().flatten())
    test_probs_sev.append(p_sev.numpy().flatten())
    test_true_cls.append(targets['classification_head'].numpy().flatten())
    test_true_sev.append(targets['severity_head'].numpy().flatten())

test_probs_cls = np.concatenate(test_probs_cls)
test_probs_sev = np.concatenate(test_probs_sev)
test_true_cls  = np.concatenate(test_true_cls).astype(int)
test_true_sev  = np.concatenate(test_true_sev)
test_preds_cls = (test_probs_cls >= OPTIMAL_THRESHOLD).astype(int)

test_acc  = np.mean(test_preds_cls == test_true_cls)
test_auc  = roc_auc_score(test_true_cls, test_probs_cls)
test_mae  = np.mean(np.abs(test_probs_sev - test_true_sev))
test_rmse = np.sqrt(np.mean((test_probs_sev - test_true_sev) ** 2))

# Severity MAE broken down by class
ctrl_mae  = np.mean(np.abs(test_probs_sev[test_true_cls==0] - test_true_sev[test_true_cls==0]))
dyslex_mae = np.mean(np.abs(test_probs_sev[test_true_cls==1] - test_true_sev[test_true_cls==1]))

print("=" * 65)
print("FINAL TEST SET RESULTS — DyslexiaLens v2")
print("=" * 65)
print(f"  Classification Accuracy  : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  AUC-ROC                  : {test_auc:.4f}")
print(f"  Severity MAE (overall)   : {test_mae:.4f}")
print(f"    ↳ Control   (class 0)  : {ctrl_mae:.4f}  ← target: ~0.00")
print(f"    ↳ Dyslexia  (class 1)  : {dyslex_mae:.4f}")
print(f"  Severity RMSE            : {test_rmse:.4f}")
print(f"  Optimal threshold        : t = {OPTIMAL_THRESHOLD:.2f}")
print("=" * 65)
print()
print("Comparison vs. baselines:")
print(f"  v1 run-1   : Acc=97.19%, Severity MAE=0.4240, t=0.66")
print(f"  Single-modal baseline : Acc=97.86%, MAE=0.0214, t=0.40")
print()
print(classification_report(test_true_cls, test_preds_cls,
                             target_names=['Control (0)', 'Dyslexia (1)']))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('DyslexiaLens v2 — Test Set Evaluation', fontsize=13, fontweight='bold')

# Confusion matrix
cm = confusion_matrix(test_true_cls, test_preds_cls)
ax = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Control', 'Dyslexia'],
            yticklabels=['Control', 'Dyslexia'])
ax.set_title(f'Confusion Matrix (t = {OPTIMAL_THRESHOLD:.2f})')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')

# Severity scatter — dyslexia only
ax = axes[1]
mask_dys = test_true_cls == 1
ax.scatter(test_true_sev[mask_dys], test_probs_sev[mask_dys],
           alpha=0.15, s=8, color='steelblue', label='Dyslexia samples')
ax.plot([0,1],[0,1], 'r--', alpha=0.7, label='Perfect prediction')
ax.set_xlabel('True Severity Score')
ax.set_ylabel('Predicted Severity Score')
ax.set_title(f'Severity — Dyslexia Samples\nMAE = {dyslex_mae:.4f}')
ax.legend(); ax.grid(alpha=0.3)

# Severity distribution by class — key diagnostic for floor fix
ax = axes[2]
ax.hist(test_probs_sev[test_true_cls==0], bins=50, alpha=0.6,
        color='steelblue', label=f'Control (n={int((test_true_cls==0).sum()):,})')
ax.hist(test_probs_sev[test_true_cls==1], bins=50, alpha=0.6,
        color='coral',    label=f'Dyslexia (n={int((test_true_cls==1).sum()):,})')
ax.axvline(0.0, color='black', linestyle='--', alpha=0.5, label='Target floor (0.0)')
ax.set_xlabel('Predicted Severity Score')
ax.set_ylabel('Count')
ax.set_title('Severity Score Distribution by Class\n(floor fix: control peak should be near 0.0)')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/test_evaluation_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → /kaggle/working/test_evaluation_v2.png")


## 15. Model Persistence

In [ ]:
model.save(MODEL_SAVE_PATH)
print(f"Full model saved  → {MODEL_SAVE_PATH}")

import pickle
with open(SCALER_SAVE_PATH, 'wb') as f:
    pickle.dump(scaler, f)
print(f"Feature scaler    → {SCALER_SAVE_PATH}")

reloaded = keras.models.load_model(
    MODEL_SAVE_PATH,
    custom_objects={
        'AdaptiveContrastNorm': AdaptiveContrastNorm,
        'MaskedHuberLoss':      MaskedHuberLoss,
    }
)
reloaded.summary(line_length=80)
print("\n✓ Model reloaded successfully (ACN + MaskedHuberLoss preserved).")

import subprocess
result = subprocess.run(['ls', '-lh', WORKING_DIR], capture_output=True, text=True)
print(f"\nFiles in {WORKING_DIR}:")
print(result.stdout)


## 16. Inference Utility Function

In [ ]:
def predict_single_sample(
    image_path: str,
    raw_features: dict,
    model=model,
    scaler=scaler,
    threshold: float = OPTIMAL_THRESHOLD
) -> dict:
    """
    Run DyslexiaLens v2 inference on a single handwriting image.

    Args:
        image_path   : Path to a .png image.
        raw_features : Dict with keys matching FEATURE_COLS (unscaled values).
        model        : Trained Keras model.
        scaler       : Fitted MinMaxScaler (from training).
        threshold    : Decision boundary (default: sweep-optimal t).

    Returns:
        dict with 'dyslexia_probability', 'predicted_class', 'label',
                  'severity_score', 'severity_level'
    """
    img = tf.io.read_file(image_path)
    img = tf.image.decode_png(img, channels=IMG_CHANNELS)
    img = tf.image.resize(img, IMG_SIZE, method='bilinear')
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.expand_dims(img, 0)

    feat_vec = np.array([[raw_features[c] for c in FEATURE_COLS]], dtype=np.float32)
    feat_vec = scaler.transform(feat_vec)
    feat_vec = tf.constant(feat_vec, dtype=tf.float32)

    p_cls, p_sev = model({'image_input': img, 'feature_input': feat_vec}, training=False)
    prob_cls = float(p_cls.numpy().flatten()[0])
    prob_sev = float(p_sev.numpy().flatten()[0])

    predicted = int(prob_cls >= threshold)
    label     = 'Dyslexia' if predicted == 1 else 'Control'

    if predicted == 0:
        sev_level = 'None'
    elif prob_sev < 0.33:
        sev_level = 'Mild'
    elif prob_sev < 0.66:
        sev_level = 'Moderate'
    else:
        sev_level = 'Severe'

    return {
        'dyslexia_probability': round(prob_cls, 4),
        'predicted_class':      predicted,
        'label':                label,
        'severity_score':       round(prob_sev if predicted == 1 else 0.0, 4),
        'severity_level':       sev_level,
    }

print("predict_single_sample() ready.")
print()
print("Example call:")
print("""
result = predict_single_sample(
    image_path='path/to/sample.png',
    raw_features={
        'stroke_density':      0.15,
        'center_of_mass_x':    13.2,
        'center_of_mass_y':    13.5,
        'bounding_box_ratio':  1.05,
        'stroke_transitions':  2.14,
        'horizontal_symmetry': 0.83,
    }
)
print(result)
""")
